In [1]:
#data inegst establish path
from google.colab import drive
try:
    drive.mount('/content/drive')
except ValueError:
    drive.flush_and_unmount()
    drive.mount('/content/drive', force_remount=True)
import os
import json
import numpy as np
import pandas as pd
from collections import Counter
from datetime import datetime,timezone
#filling the paths source, database
sourcedata = '/content/drive/MyDrive/Glioma data'
database = '/content/drive/MyDrive/Tom_HLPA'
v1sourcedata = f'{database}'
specificationpath = f'{v1sourcedata}/IngestionStage/Specification.json'
mappedgenes = f'{v1sourcedata}/IngestionStage/Genemapping.json'
#gene and ml spec sheets
with open(specificationpath) as f:
  specification = json.load(f)
with open(mappedgenes) as f:
  mappedgenes = json.load(f)
#varaiables for next cell
topgenes = 5000
logoffset = 1
extremenegatives = -100
RandomSeed = specification['discovery']['random_state']

print (RandomSeed)
print (f'source = {sourcedata}')
print (f'specification = {specification}')

Mounted at /content/drive
42
source = /content/drive/MyDrive/Glioma data
specification = {'version': '1.0', 'frozen_date': '2026-09-10', 'source_data_directory': '/content/drive/MyDrive/Glioma data', 'workspace': '/content/drive/MyDrive/HLP_Atlas_Workspace/v1.0', 'datasets': {'GLASS_GBM': {'expression': 'core_mrna.parquet', 'clinical': 'core_clin2.parquet', 'survival': 'core_ttr.parquet', 'longitudinal': True, 'modalities': ['mRNA'], 'model': 'MEFISTO', 'preprocessing': {'log_transform': 'NOT_APPLIED', 'log_transform_justification': 'Source matrix is already on a standardised scale (median per-gene std = 0.993). Applying log2(TPM+1) would be invalid.', 'duplicate_gene_handling': 'collapse by mean before selection', 'n_duplicate_rows_collapsed': 5, 'gene_filtering': 'drop all-NaN, constant, extreme-negative; keep top 5000 by variance', 'dropped_genes': {'all_nan': 487, 'constant_value': 24, 'extreme_negative': 9, 'total_dropped': 520}, 'standardisation': 'z-score per gene across samples

In [2]:
#clean and cut the data, remove all nan entries duped columns and get the highest variance genes we actually wang
def MappedGenesEsngtoHugo(expression):
    original = list(expression.columns)
    lengthoriginal = len(original)
#characterise hugo v ensg  and check the maped spec sheet to chnage them over
    SampleIDs = [str(g) for g in original[:100]]
    NumESNG = sum(1 for g in SampleIDs if g.startswith('ENSG'))
    IDtype = 'ENSG' if NumESNG > len(SampleIDs) / 2 else 'HUGO'

    NewColumns = []
    Unmapped = 0

    for g in original:
        StringG = str(g).strip()

        if StringG.startswith('ENSG'):
            BaseIsolated = StringG.split('.')[0]
            HUGO = mappedgenes.get(BaseIsolated)

            if HUGO is None:
                Unmapped += 1
                NewColumns.append(None)
            else:
                NewColumns.append(HUGO)
        else:
            NewColumns.append(StringG.upper())

    expression = expression.copy()
    expression.columns = NewColumns
    expression = expression.loc[
        :, [c is not None for c in expression.columns]
    ]
#remove duped columns
    NumDupedColumns = int(expression.columns.duplicated().sum())

    if NumDupedColumns > 0:
        expression = expression.T.groupby(expression.columns).mean().T

    return expression, {
        'IDtypefound': IDtype,
        'NumOGColumns': lengthoriginal,
        'Unmapped': Unmapped,
        'KilledDupes': NumDupedColumns,
        'FinalColumns': int(expression.shape[1]),
    }


def expressionprocessed(expression, applylog):
    expression = expression.apply(pd.to_numeric, errors='coerce')

    Sourcemin = float(np.nanmin(expression.values))
    Sourcemax = float(np.nanmax(expression.values))

    beforedrop = expression.shape[1]
    expression = expression.dropna(axis=1, how='any')
    numberofdropped = beforedrop - expression.shape[1]

    if applylog:
        expression = np.log2(expression + logoffset)
        transform = f'log2(x + {logoffset})'
    else:
        transform = 'NOT_APPLIED'

    expressionvar = expression.var(axis=0)
    genesin = len(expressionvar)
    keeped = min(topgenes, genesin)

    topgenes2 = expressionvar.nlargest(keeped).index
    expressiontop = expression[topgenes2]

    means = expressiontop.mean(axis=0)
    stddev = expressiontop.std(axis=0)
    zscores = (expressiontop - means) / stddev.replace(0, 1.0)

    return zscores, {
        'sourcevaluemin': Sourcemin,
        'sourcevaluemax': Sourcemax,
        'genesnandropped': int(numberofdropped),
        'genesbeforevariance': int(genesin),
        'geneskept': int(keeped),
        'transform': transform,
        'finalshape': list(zscores.shape),
    }

#fucntions to split up the tcga and glass patient ids
def extractTCGA(SampleIDs):
    parts = str(SampleIDs).split('-')
    return '-'.join(parts[:3]) if len(parts) >= 3 else str(SampleIDs)


def extractGLASStimepoint(SampleIDs):
    parts = str(SampleIDs).split('-')

    if len(parts) >= 4:
        return '-'.join(parts[:3]), parts[-1]

    return str(SampleIDs), 'unknown'


print('Done!')

Done!


In [ ]:
Manifestentries = []
ErrorCohorts = []
import gc
os.makedirs(database, exist_ok=True)

for cohorts in specification['datasets']['TCGA']['cohorts']:
  Dataset = f'TCGA_{cohorts}'
  SourceNow = os.path.join(sourcedata, f'{Dataset}_expression.parquet')

  if not os.path.exists(SourceNow):
      ErrorCohorts.append({'dataset': Dataset, 'error': 'source missing'})
      print(f'source mssing for {Dataset}')
      continue

  try:
      Reading = pd.read_parquet(SourceNow)
      rawshape = list(Reading.shape)
      print(f' Raw {Dataset} = {rawshape[0]} rows, {rawshape[1]},columns')

      if rawshape[0] > rawshape[1]:
        if Reading.iloc[:,0].dtype == 'object':
          genes = Reading.iloc[:,0].astype(str).values
          Reading = Reading.iloc[:,1:].T
          Reading.columns = genes
          numgenes = Reading.shape[1]
          print(numgenes)
        else:
          Reading = Reading.T
          print(f"  Transposed, ")

      print(f"  Oriented: {Reading.shape[0]} samples × {Reading.shape[1]} genes")

      Reading, harm = MappedGenesEsngtoHugo(Reading)
      print(f"  Harmonised: {harm['FinalColumns']} unique Hugo symbols")

      zscores, pp = expressionprocessed(Reading, applylog=True)

      sampleIDs67 = list(zscores.index)
      patientids = [extractTCGA(s) for s in sampleIDs67]

      meta = pd.DataFrame({
          "sampleid": sampleIDs67,
          "patientid": patientids,
          "timepoint": ["single"] * len(sampleIDs67),
      }, index=sampleIDs67)

      meta.index.name = "sample_id"
      zscores.index.name = "sample_id"

      #save
      outexpression = f'{database}/{Dataset}_expression.parquet'
      outmetadata = f'{database}/{Dataset}_metadata.parquet'

      zscores.to_parquet(outexpression)
      meta.to_parquet(outmetadata)

      #verification
      verify = pd.read_parquet(outexpression)
      assert verify.shape == zscores.shape, "shape mismatch"
      assert list(verify.columns) == list(zscores.columns), "columns mismatch"

      print(f"  Saved + verified")

      Manifestentries.append({
          "dataset": Dataset,
          "source_file": os.path.basename(SourceNow),
          "raw_shape": rawshape,
          "final_shape": list(zscores.shape),
          "n_samples": int(zscores.shape[0]),
          "n_patients": len(set(patientids)),
          "longitudinal": False,
          "harmonisation": harm,
          "preprocessing": pp,
          "output_expression": os.path.basename(outexpression),
          "output_metadata": os.path.basename(outmetadata),
      })

      del Reading, zscores, meta, verify
      gc.collect()

  except Exception as e:
      import traceback
      msg = f"{type(e).__name__}: {e}"
      print(f"{msg}")
      traceback.print_exc()
      ErrorCohorts.append({"dataset": Dataset, "error": msg})

print("i love Brian O'Connell")
print (len(Manifestentries))

 Raw TCGA_BLCA = 60660 rows, 429,columns
60660
  Oriented: 428 samples × 60660 genes
  Harmonised: 10310 unique Hugo symbols
  Saved + verified
 Raw TCGA_BRCA = 60660 rows, 1227,columns
60660
  Oriented: 1226 samples × 60660 genes
  Harmonised: 10310 unique Hugo symbols
  Saved + verified
 Raw TCGA_COAD = 60660 rows, 515,columns
60660
  Oriented: 514 samples × 60660 genes
  Harmonised: 10310 unique Hugo symbols
  Saved + verified
 Raw TCGA_KIRC = 60660 rows, 611,columns
60660
  Oriented: 610 samples × 60660 genes
  Harmonised: 10310 unique Hugo symbols
  Saved + verified
 Raw TCGA_LIHC = 60660 rows, 425,columns
60660
  Oriented: 424 samples × 60660 genes
  Harmonised: 10310 unique Hugo symbols
  Saved + verified
 Raw TCGA_LUAD = 60660 rows, 590,columns
60660
  Oriented: 589 samples × 60660 genes
  Harmonised: 10310 unique Hugo symbols
  Saved + verified
 Raw TCGA_OV = 60660 rows, 430,columns
60660
  Oriented: 429 samples × 60660 genes
  Harmonised: 10310 unique Hugo symbols
  Saved + v

In [ ]:
#GlassGBM standardisation

Dataset = "Glass GBM"
source = os.path.join(sourcedata, "core_mrna.parquet")

try:
    Reading = pd.read_parquet(source)
    rawshape = list(Reading.shape)
    print(f' Raw {Dataset} = {rawshape[0]} rows, {rawshape[1]},columns')

    matrix = Reading.iloc[:,1:].apply(pd.to_numeric, erros = 'coerce')
    matrix.index = Reading.iloc[:,0].astype(str).values
    matrix.columns = [str(c) for c in matrixcolumns]
    matrix.indexname=g0:
    enes
    print(f' size = {matrix.shape[0]}, rows {matrix.size[0]} ' )

    numberrowsbefore = len(matrix)
    nrowunique = matrix.index.nunique()
    numdupes = nrowunique - numberrowsbefore
    print(f'Dupes ={numdupes}')

    if numdupes > 0:
      matrix = matrix.groupby(level=0).mean()
    assert matrix.index.is_unique
    print(f' after = {matrix.index[1]}')

    allnan = matrix.isna().all(axis = 1)
    valuefornan = int(allnan.sum())

    def isconst(row):
      finite = row.dropna()
      return len(finite) >= 2 and finite.nunique() == 1

    constant  = matrix.apply(istconst, axis = 0)
    nconstant = int(constant.sum)

    extreme = (matrix, extremenegatives).any(axis =1)
    numberofextremes = int(extreme.sum())


    broken = allnan | constant | extreme
    n_problem = int(problem.sum())

    matrixclean = matrix.loc[~broken].copy()
    nclean = matrixclean
    clean.shape[0]

    expressionvar =matrixcleaned.var(axis=1)
    genesin = len(expressionvar)
    keeped = min(topgenes, genesin)

    topgenes2 = expressionvar.nlargest(keeped).index
    expressiontop = matrixcleaned.loc[topgenes2]

    means = expressiontop.mean(axis=1)
    stddev = expressiontop.std(axis=1)
    zscores = (expressiontop - means) / stddev.replace(0, 1.0)

    sample_ids = list(expr_final.index)
    parsed = [extractGLASStimepoint(s) for s in sample_ids]
    patient_ids = [p for p, _ in parsed]
    timepoints = [tp for _, tp in parsed]

    meta = pd.DataFrame({
        "sample_id": sample_ids,
        "patient_id": patient_ids,
        "timepoint": timepoints,
    }, index=sample_ids)
    meta.index.name = "sample_id"

    finalzscores = matrixcleaned.T
    finalzscores.index.name = 'SampleIDs'
    assert finalzsvores.columns.is_unique







_IncompleteInputError: incomplete input (1085721555.py, line 21)

In [ ]:
#GlassGBM standardisation

Dataset = "GLASS_GBM"
source = os.path.join(sourcedata, "core_mrna.parquet")

try:
    Reading = pd.read_parquet(source)
    rawshape = list(Reading.shape)
    print(f' Raw {Dataset} = {rawshape[0]} rows, {rawshape[1]} columns')

    matrix = Reading.iloc[:,1:].apply(pd.to_numeric, errors='coerce')
    matrix.index = Reading.iloc[:,0].astype(str).values
    matrix.columns = [str(c) for c in matrix.columns]
    matrix.index.name = 'gene'
    print(f' size = {matrix.shape[0]} rows, {matrix.shape[1]} samples')

    numberrowsbefore = len(matrix)
    nrowunique = matrix.index.nunique()
    numdupes = numberrowsbefore - nrowunique
    print(f'Dupes = {numdupes}')

    if numdupes > 0:
        matrix = matrix.groupby(level=0).mean()

    assert matrix.index.is_unique
    print(f' after = {matrix.shape[0]} unique genes')

    allnan = matrix.isna().all(axis=1)
    valuefornan = int(allnan.sum())

    def isconst(row):
        finite = row.dropna()
        return len(finite) >= 2 and finite.nunique() == 1

    constant = matrix.apply(isconst, axis=1)
    nconstant = int(constant.sum())

    extreme = (matrix < extremenegatives).any(axis=1)
    numberofextremes = int(extreme.sum())

    broken = allnan | constant | extreme
    n_problem = int(broken.sum())

    matrixclean = matrix.loc[~broken].copy()
    nclean = matrixclean.shape[0]

    print(f' QC drop: all-NaN={valuefornan}, constant={nconstant}, extreme-neg={numberofextremes} (total {n_problem})')
    print(f' Clean genes: {nclean}')

    expressionvar = matrixclean.var(axis=1)
    genesin = len(expressionvar)
    keeped = min(topgenes, genesin)

    topgenes2 = expressionvar.nlargest(keeped).index
    expressiontop = matrixclean.loc[topgenes2]

    assert expressiontop.index.is_unique

    means = expressiontop.mean(axis=1)
    stddev = expressiontop.std(axis=1)
    zscores = expressiontop.sub(means, axis=0).div(stddev.replace(0, 1.0), axis=0)

    print(f' Selected top {keeped} by variance')
    print(f' Z-scored: {zscores.shape}')

    expr_final = zscores.T
    expr_final.index.name = 'sample_id'
    assert expr_final.columns.is_unique

    sample_ids = list(expr_final.index)
    parsed = [extractGLASStimepoint(s) for s in sample_ids]
    patient_ids = [p for p, _ in parsed]
    timepoints = [tp for _, tp in parsed]

    meta = pd.DataFrame({
        "sample_id": sample_ids,
        "patient_id": patient_ids,
        "timepoint": timepoints,
    }, index=sample_ids)
    meta.index.name = "sample_id"

    outexpression = f'{database}/{Dataset}_expression.parquet'
    outmetadata = f'{database}/{Dataset}_metadata.parquet'

    expr_final.to_parquet(outexpression)
    meta.to_parquet(outmetadata)

    verify = pd.read_parquet(outexpression)
    assert verify.shape == expr_final.shape, "shape mismatch"
    assert list(verify.columns) == list(expr_final.columns), "columns mismatch"

    print(f' Saved + verified')

    pt_counts = Counter(patient_ids)
    n_longitudinal = sum(1 for c in pt_counts.values() if c > 1)
    print(f' Patients: {len(pt_counts)}, longitudinal: {n_longitudinal}')
    print(f' Timepoints: {dict(Counter(timepoints))}')

    Manifestentries.append({
        "dataset": Dataset,
        "source_file": "core_mrna.parquet",
        "raw_shape": rawshape,
        "final_shape": list(expr_final.shape),
        "n_samples": int(expr_final.shape[0]),
        "n_patients": len(pt_counts),
        "n_longitudinal_patients": n_longitudinal,
        "timepoint_distribution": dict(Counter(timepoints)),
        "longitudinal": True,
        "duplicate_collapse": {
            "n_rows_before": int(numberrowsbefore),
            "n_duplicates": int(numdupes),
            "n_after_collapse": int(matrix.shape[0]),
        },
        "gene_qc": {
            "all_nan_dropped": valuefornan,
            "constant_dropped": nconstant,
            "extreme_negative_dropped": numberofextremes,
            "n_clean": int(nclean),
            "n_selected": int(keeped),
        },
        "preprocessing": {
            "log_transform": "NOT_APPLIED",
            "log_justification": "Source matrix is already on standardised scale (median per-gene std = 0.993)",
            "standardisation": "z-score per gene",
        },
        "output_expression": os.path.basename(outexpression),
        "output_metadata": os.path.basename(outmetadata),
    })

    del Reading, matrix, matrixclean, expressiontop, zscores, expr_final, meta, verify
    gc.collect()

except Exception as e:
    import traceback
    msg = f"{type(e).__name__}: {e}"
    print(f'{msg}')
    traceback.print_exc()
    ErrorCohorts.append({"dataset": Dataset, "error": msg})